In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
--- ISRC MATCHING 
CREATE OR REPLACE TABLE ISRC_WITH_WORKS_TRACKS_32X AS (
WITH DISTINCT_TITLE_MATCHES AS (
    SELECT DISTINCT 
           APRA_WORK_ID,
           MUZOOKA_TRACK_ID,
           RDC_WORKS_ID
    FROM ADC_WORKS_COLUMNS_TITLE_MATCHES_COMBINED_32X
),
CLEANED_ISRC AS (
    SELECT 
        c.*,
        i.ADC_ISRC_ID,
        i.APRA_WORK_ID as i_APRA_WORK_ID,
        LEFT(REPLACE(REPLACE(TRIM(i.ISRC), '-', ''), ' ', ''), 12) as CLEANED_I_ISRC,
        LEFT(REPLACE(REPLACE(TRIM(r.isrc), '-', ''), ' ', ''), 12) as CLEANED_R_ISRC,
        i.ISRC as ORIGINAL_ISRC
    FROM 
        DISTINCT_TITLE_MATCHES c
        INNER JOIN ADCISRC i ON i.APRA_WORK_ID = c.APRA_WORK_ID
        INNER JOIN recordings r ON c.MUZOOKA_TRACK_ID = UPPER(r.track_id)
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY ADC_ISRC_ID) AS RDC_ISRC_ID,
    RDC_WORKS_ID,
    APRA_WORK_ID,
    MUZOOKA_TRACK_ID,
    CLEANED_I_ISRC as ISRC,
    CASE WHEN CLEANED_R_ISRC = CLEANED_I_ISRC THEN 'Y' ELSE 'N' END AS YN_ISRC_MATCH
FROM CLEANED_ISRC
WHERE CLEANED_R_ISRC = CLEANED_I_ISRC  -- Filter for matching ISRCs only
--WHERE APRA_WORK_ID = 'GW33624310'
);